In [166]:
# # Internal wave example (NetCDF + total buoyancy)
#
# Changes vs your original:
# 1) Use NetCDFWriter to save output; use NCDatasets to read/plot.
# 2) Output total buoyancy b_total = b + N^2 z and plot its contours.

using Oceananigans
using Oceananigans.OutputWriters: NetCDFWriter
using CairoMakie
using NCDatasets

# -----------------------------
# Domain and model definition
# -----------------------------
grid = RectilinearGrid(size=(256, 256), x=(-π, π), z=(-π, π), topology=(Periodic, Flat, Bounded))

coriolis = FPlane(f=0.0)
N = 1.0                     # buoyancy frequency [s^-1]
B̄_func(x, z, t, N) = N^2 * z
B̄ = BackgroundField(B̄_func; parameters=N)

model = NonhydrostaticModel(; grid, coriolis,
    advection = WENO(),
    closure = ScalarDiffusivity(ν=1e-6, κ=1e-6),
    tracers = :b,
    buoyancy = BuoyancyTracer(),
    background_fields = (; background_closure_fluxes=true, b = B̄)
)

# -----------------------------
# Internal wave initial state
# -----------------------------
m = 8      # vertical wavenumber
k = 8       # horizontal wavenumber
f = coriolis.f
ω² = (N^2 * k^2 + f^2 * m^2) / (k^2 + m^2)
ω = sqrt(ω²)

s = 2 # wave steepness -- CHANGE FROM <1 TO >1 FOR INSTABILITY!
displacement_amplitude = s/m # units of m
w_amplitude = displacement_amplitude * ω
pressure_amplitude = w_amplitude / (m * ω * N^2 / (N^2 - ω^2)) # Φ₀ in Vallis notation
gaussian_width = grid.Lx / 10
Φ(x, z) = pressure_amplitude * exp(-(x^2 + z^2) / (2gaussian_width^2))

eps = 1e-3 # noise amplitude
u₀(x, z) =   Φ(x, z) * k * ω   / (ω^2 - f^2) * cos(k * x + m * z)
v₀(x, z) =   Φ(x, z) * k * f   / (ω^2 - f^2) * sin(k * x + m * z)
w₀(x, z) = - Φ(x, z) * m * ω   / (N^2 - ω^2) * cos(k * x + m * z) + eps*w_amplitude*randn()
b₀(x, z) = - Φ(x, z) * m * N^2 / (N^2 - ω^2) * sin(k * x + m * z)

set!(model, u=u₀, v=v₀, w=w₀, b=b₀)

b = model.tracers.b
B̄ = model.background_fields.tracers.b
B = B̄ + b # total buoyancy field

# -----------------------------
# Simulation + NetCDF output
# -----------------------------
simulation = Simulation(model; Δt = 0.002 * 2π/ω, stop_iteration = 2000)

Simulation of NonhydrostaticModel{CPU, RectilinearGrid}(time = 0 seconds, iteration = 0)
├── Next time step: 17.772 ms
├── Elapsed wall time: 0 seconds
├── Wall time per iteration: NaN days
├── Stop time: Inf days
├── Stop iteration: 2000.0
├── Wall time limit: Inf
├── Minimum relative step: 0.0
├── Callbacks: OrderedDict with 4 entries:
│   ├── stop_time_exceeded => Callback of stop_time_exceeded on IterationInterval(1)
│   ├── stop_iteration_exceeded => Callback of stop_iteration_exceeded on IterationInterval(1)
│   ├── wall_time_limit_exceeded => Callback of wall_time_limit_exceeded on IterationInterval(1)
│   └── nan_checker => Callback of NaNChecker for u on IterationInterval(100)
├── Output writers: OrderedDict with no entries
└── Diagnostics: OrderedDict with no entries

In [167]:
exp_name = string("internal_wave_packet_s=", s)
ncfile = string("../data/raw_simulation_output/", exp_name, ".nc")
simulation.output_writers[:fields] = NetCDFWriter(
    model,
    (
        w = model.velocities.w,    # vertical velocity
        B = B          # total buoyancy b + N^2 z
    ),
    filename = ncfile,
    schedule = IterationInterval(10),
    overwrite_existing = true,
)

run!(simulation)

[ Info: Initializing simulation...
[ Info:     ... simulation initialization complete (341.671 ms)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (57.591 ms).
[ Info: Simulation is stopping after running for 0 seconds.
[ Info: Model iteration 2000 equals or exceeds stop iteration 2000.


In [168]:
using NCDatasets
using CairoMakie

set_theme!(Theme(fontsize = 20))

# --- Open dataset ---
ds = NCDataset(simulation.output_writers[:fields].filepath, "r")
x  = ds["x_caa"][:]         # x cell centers
zf = ds["z_aaf"][:]         # z faces
zc = ds["z_aac"][:]         # z centers
t  = ds["time"][:]          # time (s)

# --- Observables for animation ---
n = Observable(1)
w_plot = @lift ds["w"][:, :, $n]
B_plot = @lift ds["B"][:, :, $n]

# --- Figure + axes ---
fig = Figure(size = (880, 720))  # a bit wider for the colorbar
ax  = Axis(fig[2, 1];
    xlabel = "x", ylabel = "z",
    limits = ((minimum(x), maximum(x)), (minimum(zf), maximum(zf))),
    aspect = AxisAspect(1),
)

# --- Titles / annotations ---
title_lbl = Label(fig[1, 1], @lift("t = $(round(t[$n], digits=2))"),
                  fontsize=24, tellwidth=false)
annot_lbl = Label(fig[3, 1], "wave steepness s = $(round(s, digits=3))",
                  fontsize=18, tellwidth=false)

# --- Field visualization ---
w_lim    = 2 * w_amplitude                          # requested range: [-2w_amp, 2w_amp]
w_levels = LinRange(-w_lim, w_lim, 41)              # 21 levels gives 20 bins

w_img = contourf!(ax, x, zf, w_plot;
    levels = w_levels,
    colormap = :balance,
    extendlow = :auto,
    extendhigh = :auto
)

# Buoyancy contours overlay
contour!(ax, x, zc, B_plot;
    levels = LinRange(-π, π, 25),
    color = :black, linewidth = 1
)

# --- Colorbar (linked to w_img) ---
Colorbar(fig[2, 2], w_img;
    label = "Vertical velocity w",
    width = 15,
    ticklabelsize = 16,
    labelsize = 18
)

# --- Animate ---
frames = 1:length(t)
record(fig, string("../movies/", exp_name, ".mp4"), frames; framerate = 12) do i
    n[] = i
end

close(ds)

closed Dataset